In [ ]:
import pandas as pd
import numpy as np
import glob
import os
'''
1, 3, 5, 15분봉 데이터 전처리용 
기존 전처리 코드에 차분 데이터 추가 작성 
'''


# 1. 기술적 지표 계산 함수 (기존과 동일)
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)

def calculate_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    macd_hist = macd_line - signal_line
    return macd_line, signal_line, macd_hist

def calculate_bollinger(series, period=20, std_dev=2):
    ma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    upper = ma + (std * std_dev)
    lower = ma - (std * std_dev)
    width = (upper - lower) / (ma.replace(0, np.nan))
    pct = (series - lower) / (upper - lower)
    return upper, ma, lower, width, pct

def calculate_atr(df, period=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    return atr

def get_weights_ffd(d, thres=1e-4):
    """
    Lopez de Prado식 FFD(weight) 생성.
    d: fractional order (예: 0.3~0.5)
    thres: weight 절대값이 이 값보다 작아지면 더 이상 추가하지 않음
    """
    w = [1.0]
    k = 1
    while True:
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < thres:
            break
        w.append(w_k)
        k += 1
    w = np.array(w[::-1]).reshape(-1, 1)  # 최근 값이 맨 아래 오도록 역순
    return w

def frac_diff_ffd(series, d, thres=1e-4):
    """
    Fixed-width FFD (Fractional Differencing).
    과거 값만 사용하는 형태라 미래누수 없음.
    """
    series = series.astype('float64').dropna()
    w = get_weights_ffd(d, thres)
    width = len(w)

    out = pd.Series(index=series.index, dtype='float64')

    for i in range(width - 1, len(series)):
        window = series.iloc[i - width + 1 : i + 1].values
        out.iloc[i] = np.dot(w.T, window)[0]

    return out

def add_indicators(df):
    df = df.copy()
    # 이동평균선
    for w in [5, 10, 20, 60, 120]:
        df[f'sma_{w}'] = df['close'].rolling(window=w).mean()

    df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()
    df['rsi_14'] = calculate_rsi(df['close'], 14)
    df['macd'], df['macd_sig'], df['macd_hist'] = calculate_macd(df['close'])
    _, df['bb_mid'], _, df['bb_width'], df['bb_pct'] = calculate_bollinger(df['close'])
    df['atr_14'] = calculate_atr(df)
    df['vol_ma_20'] = df['volume'].rolling(window=20).mean()

    # 수익률 및 변동성
    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    df['volatility'] = df['log_return'].rolling(window=20).std()
    return df

def add_cyclical_features(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    return df

def resample_data(df, interval):
    # 명시적으로 closed='left', label='left' 설정 (기본값이지만 명확히 함)
    # 예: 15:00:00 ~ 15:14:59 데이터는 '15:00:00' 라벨로 묶임
    agg_dict = {
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum',
        'value': 'sum'
    }
    resampled = df.resample(interval, closed='left', label='left').agg(agg_dict).dropna()
    return resampled

def merge_higher_tf(original_df, higher_df, suffix):
    # 병합할 컬럼 선택 (OHLCV 제외한 지표들)
    cols_to_merge = [c for c in higher_df.columns if c not in ['open', 'high', 'low', 'close', 'volume', 'value']]
    subset = higher_df[cols_to_merge].copy()

    # 핵심: 미래 참조(Data Leakage) 방지를 위한 Shift
    subset = subset.shift(1)

    # 컬럼명 변경 (예: rsi_14 -> rsi_14_15m)
    subset.columns = [f"{c}_{suffix}" for c in subset.columns]

    # 원본(1분봉) 인덱스에 맞춰 데이터 채우기 (ffill: 15:15 값으로 15:16~15:29 채움)
    subset_reindexed = subset.reindex(original_df.index, method='ffill')

    return pd.concat([original_df, subset_reindexed], axis=1)

# 2. 데이터 처리 파이프라인
INPUT_FILE = '/content/drive/MyDrive/btc_trading_bot/KRW-BTC_1min.csv'
OUTPUT_FILE = '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv'

if os.path.exists(INPUT_FILE):
    df_1m = pd.read_csv(INPUT_FILE)
else:
    raise FileNotFoundError(f"{INPUT_FILE} 파일이 없습니다. 경로를 확인해주세요.")

# 인덱스 설정
if 'index' in df_1m.columns:
    df_1m['datetime'] = pd.to_datetime(df_1m['index'])
elif 'datetime' in df_1m.columns:
    df_1m['datetime'] = pd.to_datetime(df_1m['datetime'])
df_1m.set_index('datetime', inplace=True)
if 'index' in df_1m.columns:
    df_1m.drop('index', axis=1, inplace=True)

df_1m = df_1m[~df_1m.index.duplicated(keep='first')].sort_index()
print(f"원본 데이터 로드 완료: {df_1m.shape}")

# 상위 시간봉 생성
df_3m = resample_data(df_1m, '3min')
df_5m = resample_data(df_1m, '5min')
df_15m = resample_data(df_1m, '15min')

# 기술적 지표 추가
df_1m = add_indicators(df_1m)
df_3m = add_indicators(df_3m)
df_5m = add_indicators(df_5m)
df_15m = add_indicators(df_15m) # 15분봉 지표 계산

# 시간 피처 추가
df_1m = add_cyclical_features(df_1m)

# 데이터 병합 (3m, 5m, 15m 모두 shift(1) 적용됨)
df_final = merge_higher_tf(df_1m, df_3m, '3m')
df_final = merge_higher_tf(df_final, df_5m, '5m')
df_final = merge_higher_tf(df_final, df_15m, '15m') # 여기서 15분봉도 누수 방지 적용됨

df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)

print(f"전처리 완료 데이터 형태 (지표/상위TF 병합 후): {df_final.shape}")

# ============================================================
# NEW 1) 분수 차분 피처 추가 (close + 주요 인디케이터)
# ============================================================

FRAC_D = 0.4      # 분수 차분 차수 (0.3~0.5 권장)
THRES  = 1e-4     # weight 컷오프

# 분수 차분을 적용할 주요 컬럼 리스트
frac_cols = [
    'close',
    'sma_5', 'sma_10', 'sma_20',
    'ema_12', 'ema_26',
    'rsi_14',
    'macd',
    'bb_pct',
    'volatility',
]

# 실제 존재하는 컬럼만 필터
frac_cols = [c for c in frac_cols if c in df_final.columns]

for col in frac_cols:
    fd_series = frac_diff_ffd(df_final[col], d=FRAC_D, thres=THRES)
    df_final[f'{col}_fd{FRAC_D}'] = fd_series
    print(f"Fractional diff added: {col} -> {col}_fd{FRAC_D}")

# 분수 차분 추가 후 앞부분 NaN 줄 정리
df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)
print(f"분수 차분 피처 추가 후 데이터 형태: {df_final.shape}")

원본 데이터 로드 완료: (518546, 6)
전처리 완료 데이터 형태 (지표/상위TF 병합 후): (516746, 82)
Fractional diff added: close -> close_fd0.4
Fractional diff added: sma_5 -> sma_5_fd0.4
Fractional diff added: sma_10 -> sma_10_fd0.4
Fractional diff added: sma_20 -> sma_20_fd0.4
Fractional diff added: ema_12 -> ema_12_fd0.4
Fractional diff added: ema_26 -> ema_26_fd0.4
Fractional diff added: rsi_14 -> rsi_14_fd0.4
Fractional diff added: macd -> macd_fd0.4
Fractional diff added: bb_pct -> bb_pct_fd0.4
Fractional diff added: volatility -> volatility_fd0.4
분수 차분 피처 추가 후 데이터 형태: (516465, 92)


In [ ]:
import pandas as pd
import numpy as np
import glob
import os
'''
단기의 경우 수수료의 문제에 빠지는 것을 피하기 위하여 장기 데이터로 대체

'''

# RSI지표 계산함수 
def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    loss = (-delta.where(delta < 0, 0)).ewm(alpha=1/period, adjust=False).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi.fillna(50)

# MACD지표
def calculate_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    macd_hist = macd_line - signal_line
    return macd_line, signal_line, macd_hist

# 볼린저 벤드 
def calculate_bollinger(series, period=20, std_dev=2):
    ma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    upper = ma + (std * std_dev)
    lower = ma - (std * std_dev)
    width = (upper - lower) / (ma.replace(0, np.nan))
    pct = (series - lower) / (upper - lower)
    return upper, ma, lower, width, pct

# ATR
def calculate_atr(df, period=14):
    high_low = df['high'] - df['low']
    high_close = np.abs(df['high'] - df['close'].shift())
    low_close = np.abs(df['low'] - df['close'].shift())
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    atr = tr.rolling(window=period).mean()
    return atr

# 차분 weight 계산
def get_weights_ffd(d, thres=1e-4): # 기준 0.0004
    w = [1.0]
    k = 1
    while True:
        w_k = -w[-1] * (d - k + 1) / k
        if abs(w_k) < thres:
            break
        w.append(w_k)
        k += 1
    w = np.array(w[::-1]).reshape(-1, 1)
    return w

def frac_diff_ffd(series, d, thres=1e-4):
    series = series.astype('float64').dropna()
    w = get_weights_ffd(d, thres)
    width = len(w)
    out = pd.Series(index=series.index, dtype='float64')
    for i in range(width - 1, len(series)):
        window = series.iloc[i - width + 1 : i + 1].values
        out.iloc[i] = np.dot(w.T, window)[0]
    return out

# 대부분의 EMA지표가 상관계수가 1로 나타남 
# EMA지표는 삭제하는 것이 유용해 보임 (현재 모델은 포함되어있으므로 수정 )
def add_indicators(df):
    df = df.copy()
    for w in [5, 10, 20, 60, 120]:
        df[f'sma_{w}'] = df['close'].rolling(window=w).mean()

    df['ema_12'] = df['close'].ewm(span=12, adjust=False).mean()
    df['ema_26'] = df['close'].ewm(span=26, adjust=False).mean()
    df['rsi_14'] = calculate_rsi(df['close'], 14)
    df['macd'], df['macd_sig'], df['macd_hist'] = calculate_macd(df['close'])
    _, df['bb_mid'], _, df['bb_width'], df['bb_pct'] = calculate_bollinger(df['close'])
    df['atr_14'] = calculate_atr(df)
    df['vol_ma_20'] = df['volume'].rolling(window=20).mean()

    df['log_return'] = np.log(df['close'] / df['close'].shift(1))
    df['volatility'] = df['log_return'].rolling(window=20).std()
    return df

def add_cyclical_features(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)
    return df

def resample_data(df, interval):
    agg_dict = {
        'open': 'first',
        'high': 'max',
        'low': 'min',
        'close': 'last',
        'volume': 'sum',
        'value': 'sum'
    }
    resampled = df.resample(interval, closed='left', label='left').agg(agg_dict).dropna()
    return resampled

def merge_higher_tf(original_df, higher_df, suffix):
    cols_to_merge = [c for c in higher_df.columns if c not in ['open', 'high', 'low', 'close', 'volume', 'value']]
    subset = higher_df[cols_to_merge].copy()
    subset = subset.shift(1)  # 미래 참조 방지
    subset.columns = [f"{c}_{suffix}" for c in subset.columns]
    subset_reindexed = subset.reindex(original_df.index, method='ffill')
    return pd.concat([original_df, subset_reindexed], axis=1)

# 2. 데이터 처리 파이프라인
INPUT_FILE = '/content/drive/MyDrive/btc_trading_bot/KRW-BTC_1min.csv'
OUTPUT_FILE = '/content/drive/MyDrive/btc_trading_bot/df_cleaned.csv'

if os.path.exists(INPUT_FILE):
    df_1m = pd.read_csv(INPUT_FILE)
else:
    raise FileNotFoundError(f"{INPUT_FILE} 파일이 없습니다. 경로를 확인해주세요.")

if 'index' in df_1m.columns:
    df_1m['datetime'] = pd.to_datetime(df_1m['index'])
elif 'datetime' in df_1m.columns:
    df_1m['datetime'] = pd.to_datetime(df_1m['datetime'])
df_1m.set_index('datetime', inplace=True)
if 'index' in df_1m.columns:
    df_1m.drop('index', axis=1, inplace=True)

df_1m = df_1m[~df_1m.index.duplicated(keep='first')].sort_index()
print(f"원본 데이터 로드 완료: {df_1m.shape}")


# 기존 단기 데이터에서, 장기 데이터로 변환하여 사용 
print("상위 타임프레임 리샘플링 중...")
df_15m = resample_data(df_1m, '15min')  # 15분봉
df_30m = resample_data(df_1m, '30min')  # 30분봉
df_1h  = resample_data(df_1m, '1h')     # 1시간봉
df_4h  = resample_data(df_1m, '4h')     # 4시간봉
df_6h  = resample_data(df_1m, '6h')     # 6시간봉

# 기술적 지표 추가
print("지표 계산 중...")
df_1m = add_indicators(df_1m)
df_15m = add_indicators(df_15m)
df_30m = add_indicators(df_30m)
df_1h = add_indicators(df_1h)
df_4h = add_indicators(df_4h)
df_6h = add_indicators(df_6h)

# 시간 피처 추가
df_1m = add_cyclical_features(df_1m)

print("멀티 타임프레임 병합 중...")
df_final = merge_higher_tf(df_1m, df_15m, '15m')  # 15분봉 병합
df_final = merge_higher_tf(df_final, df_30m, '30m')  # 30분봉 병합
df_final = merge_higher_tf(df_final, df_1h, '1h')    # 1시간봉 병합
df_final = merge_higher_tf(df_final, df_4h, '4h')    # 4시간봉 병합
df_final = merge_higher_tf(df_final, df_6h, '6h')    # 6시간봉 병합

df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)

print(f"전처리 완료 데이터 형태 (지표/상위TF 병합 후): {df_final.shape}")

# ============================================================
# 분수 차분 피처 추가
# ============================================================
FRAC_D = 0.4
THRES  = 1e-4

frac_cols = [
    'close',
    'sma_5', 'sma_10', 'sma_20',
    'ema_12', 'ema_26',
    'rsi_14',
    'macd',
    'bb_pct',
    'volatility',
]

frac_cols = [c for c in frac_cols if c in df_final.columns]

print("분수 차분 피처 생성 중...")
for col in frac_cols:
    fd_series = frac_diff_ffd(df_final[col], d=FRAC_D, thres=THRES)
    df_final[f'{col}_fd{FRAC_D}'] = fd_series

df_final.replace([np.inf, -np.inf], np.nan, inplace=True)
df_final.dropna(inplace=True)
print(f"분수 차분 피처 추가 후 최종 데이터 형태: {df_final.shape}")

df_final.to_csv(OUTPUT_FILE)
print(f"파일 저장 완료: {OUTPUT_FILE}")


원본 데이터 로드 완료: (518546, 6)
상위 타임프레임 리샘플링 중...
지표 계산 중...
멀티 타임프레임 병합 중...
전처리 완료 데이터 형태 (지표/상위TF 병합 후): (475706, 118)
분수 차분 피처 생성 중...
분수 차분 피처 추가 후 최종 데이터 형태: (475425, 128)
파일 저장 완료: /content/drive/MyDrive/btc_trading_bot/df_cleaned.csv


In [ ]:
print(f"최종 전처리 데이터 형태: {df_final.shape}")
df_final.to_csv(OUTPUT_FILE)
print(f"저장 완료: {OUTPUT_FILE}")

최종 전처리 데이터 형태: (516465, 92)
저장 완료: /content/drive/MyDrive/btc_trading_bot/df_cleaned.csv


In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/btc_trading_bot/df_1m_cleaned_features.csv",
    parse_dates=["datetime"]   # datetime 컬럼 파싱 필수
)

# 2. datetime 기준으로 정렬 (시계열 필수)
print(df.info())
'''
# 3. 전체 데이터의 1/50만큼 앞에서 자르기
sample_size = len(df) // 50
df_sample = df.iloc[:sample_size]

# 4. 결과 저장
df_sample.to_csv(
    "/content/drive/MyDrive/btc_trading_bot/minute_data_sample.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"총 {len(df)}개 중 {len(df_sample)}개(1/50)를 저장 완료했습니다.")
'''

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 516746 entries, 0 to 516745
Data columns (total 83 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   datetime        516746 non-null  datetime64[ns]
 1   open            516746 non-null  float64       
 2   high            516746 non-null  float64       
 3   low             516746 non-null  float64       
 4   close           516746 non-null  float64       
 5   volume          516746 non-null  float64       
 6   value           516746 non-null  float64       
 7   sma_5           516746 non-null  float64       
 8   sma_10          516746 non-null  float64       
 9   sma_20          516746 non-null  float64       
 10  sma_60          516746 non-null  float64       
 11  sma_120         516746 non-null  float64       
 12  ema_12          516746 non-null  float64       
 13  ema_26          516746 non-null  float64       
 14  rsi_14          516746 non-null  flo

'\n# 3. 전체 데이터의 1/50만큼 앞에서 자르기\nsample_size = len(df) // 50\ndf_sample = df.iloc[:sample_size]\n\n# 4. 결과 저장\ndf_sample.to_csv(\n    "/content/drive/MyDrive/btc_trading_bot/minute_data_sample.csv",\n    index=False,\n    encoding="utf-8-sig"\n)\n\nprint(f"총 {len(df)}개 중 {len(df_sample)}개(1/50)를 저장 완료했습니다.")\n'